# motion-detect — demo

Thin end-to-end demo: load the YAML config, run production inference on one clip, show the result.
Requires a checkpoint (`checkpoints/best_unet.pth`) and videos configured in `configs/paths.yaml`.


In [ ]:
from pathlib import Path
from motion_detect.config import load_config

cfg = load_config("configs/paths.yaml", defaults_path="configs/default.yaml")
cfg


In [ ]:
import torch
from motion_detect.inference import load_frozen_pytorch_backend, predict_video_bboxes
from motion_detect.utils import index_video_files

checkpoint = cfg.checkpoints_dir / "best_unet.pth"
backend = load_frozen_pytorch_backend(str(checkpoint))  # fused Conv+BN inference model
video_path = index_video_files(cfg.videos_dir)[41]

predictions = predict_video_bboxes(
    str(video_path),
    params=dict(cfg.detection_params),
    unet_model=backend.model,
    unet_threshold=cfg.unet_threshold,
    compensation_step=cfg.compensation_step,
    compensation_config=dict(cfg.compensation),
    return_size=cfg.return_size,
    apply_validity_mask=cfg.apply_validity_mask,
    apply_platform_suppression=cfg.apply_platform_suppression,
    max_width=cfg.max_width,
)
sum(len(v) for v in predictions.values()), "boxes over", len(predictions), "frames"


In [ ]:
from motion_detect.postprocess import render_predictions_video

out = (cfg.results_dir / "render" / (video_path.stem + "_annotated.mp4"))
render_predictions_video(str(video_path), str(out), predictions, output_size=cfg.return_size)
out


In [ ]:
# Show the annotated video inline (Jupyter only)
try:
    from IPython.display import Video, display
    display(Video(str(out), embed=True))
except Exception as exc:
    print("inline playback unavailable:", exc)
